In [1]:
import pandas as pd
import numpy as np 
import nnresample 
import os 
from IPython.display import Audio
import pickle

## Open timit dataframes and get word speaker noise vocabulary

In [8]:
word_and_speaker_encodings = pickle.load( open( "/om4/group/mcdermott/user/jfeather/projects/model_metamers/figure_generation_notebooks/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']


In [3]:
path= '/om4/group/mcdermott/user/raygon/home/projects/public/jsinDataset/assets/data/interim/timit/'
timit_df = pd.read_pickle(os.path.join(path,'timitWordsDataframe_interim.pdpkl'))

In [4]:
timit_df.head()

,start_sample,end_sample,word,_full_dataset_index,corpus,path,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,start_ms,end_ms
0,3050,5723,she,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,2673,3050,41074,0.190625,0.357687
1,5723,10337,had,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4614,5723,36460,0.357687,0.646062
2,9190,11517,your,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,2327,9190,35280,0.574375,0.719812
3,11517,16334,dark,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4817,11517,30463,0.719812,1.020875
4,16334,21199,suit,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4865,16334,25598,1.020875,1.324938


In [5]:
## All timit data 

padas_path = '/om4/group/mcdermott/user/raygon/home/projects/public/jsinDataset/assets/data/raw/timit/dataframes/'
pth = 'all_timit_data.pdpkl'

all_timit = pd.read_pickle(os.path.join(padas_path, pth))

In [6]:
all_timit.head()

,corpus,source,speaker_id,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,source_int,dialect_region_int,...,speaker_sex_int,sentence_type_int,sentence_id_int,sr,signal,phoneme,word,sentence,path,signal_length
0,timit,train-dr1-fcjf0-sa1,cjf0,f,sa,1,dr1,train,1680,0,...,0,0,0,16000,"[[3.051850947599719e-05, -3.051850947599719e-0...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,46797
1,timit,train-dr1-fcjf0-sa2,cjf0,f,sa,2,dr1,train,1681,0,...,0,0,1111,16000,"[[-3.051850947599719e-05, 0.0, 3.0518509475997...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,start_sa...,/home/raygon/deepFerret/data/sources/speech/ti...,34509
2,timit,train-dr1-fcjf0-si1027,cjf0,f,si,1027,dr1,train,1682,0,...,0,1,32,16000,"[[0.00024414807580797754, 0.000152592547379985...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,49460
3,timit,train-dr1-fcjf0-si1657,cjf0,f,si,1657,dr1,train,1683,0,...,0,1,731,16000,"[[0.00012207403790398877, 9.155552842799158e-0...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,45466
4,timit,train-dr1-fcjf0-si648,cjf0,f,si,648,dr1,train,1684,0,...,0,1,1952,16000,"[[-6.103701895199438e-05, 0.000183111056855983...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,57856


### Get set of TIMIT words included in vocabulary

In [8]:
all_words = np.unique(np.concatenate([all_timit.word[ix].word.values for ix in range(all_timit.shape[0])]))

In [9]:
# get list of timit words included in model vocabulary 

valid_timit_words = all_words[np.in1d(all_words, list(class_map.values()))] 

### Need to parse through timit to get excerpts with valid words 

Here we parse the dataset for 2 coniditions:

1. the words occur such that we can make 2 second excerpts centered on the word
2. the word is in our vocabulary

For 1, we crudely make sure the word either overlaps the middle of the parent excerpt or that there is at least half a second context before word onset and after word offset (given a 2 second window). 


In [262]:
## Get good ixs by screening words and word timing 


def screen_row(row):
    mid_dur = row.signal_length//2
    mid_of_2 = row.start_sample >= 8000 and row.end_sample <= 24000
    mid_of_excerpt = row.start_sample <= mid_dur and row.end_sample > mid_dur
    long_enough = mid_of_2 or mid_of_excerpt 
    return long_enough and row.word in valid_timit_words

good_timit_excerpts = timit_df[['start_sample', 'end_sample', 'word', 'signal_length']].apply(lambda x: screen_row(x), axis=1)

In [263]:
good_timit.shape

(4272, 17)

In [264]:
good_timit = timit_df[good_timit_excerpts].copy()

In [265]:
good_timit.head()

,start_sample,end_sample,word,_full_dataset_index,corpus,path,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,start_ms,end_ms
3,12291,15325,money,3,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-si1657,fcjf0,16000,45466,2841.6250,1,3034,12291,30141,0.768188,0.957812
3,13785,17545,their,6,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sx217,fcjf0,16000,27751,1734.4375,1,3760,13785,10206,0.861563,1.096563
1,11640,23170,programs,15,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdaw0-sx146,fdaw0,16000,42087,2630.4375,1,11530,11640,18917,0.727500,1.448125
4,22327,28680,novel,17,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdaw0-sx326,fdaw0,16000,45978,2873.6250,1,6353,22327,17298,1.395438,1.792500
2,10040,15382,power,18,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdaw0-sx416,fdaw0,16000,41780,2611.2500,1,5342,10040,26398,0.627500,0.961375


In [266]:
# must be 2 seconds 

good_timit = good_timit[good_timit.signal_length >= 32000]

In [281]:
good_timit.shape

(1527, 17)

### Make new df with wanted attributes 

* updampled wav (16kHz -> 20kHz)
* word label
* speaker label
* speaker sex 
* speaker dialect

In [282]:
all_timit.columns

Index(['corpus', 'source', 'speaker_id', 'speaker_sex', 'sentence_type',
       'sentence_id', 'dialect_region', 'data_split', 'source_int',
       'dialect_region_int', 'data_split_int', 'speaker_id_int',
       'speaker_sex_int', 'sentence_type_int', 'sentence_id_int', 'sr',
       'signal', 'phoneme', 'word', 'sentence', 'path', 'signal_length'],
      dtype='object')

In [283]:
good_timit.columns

Index(['start_sample', 'end_sample', 'word', '_full_dataset_index', 'corpus',
       'path', 'source', 'speaker', 'sr', 'signal_length', 'duration_ms',
       'n_channels', 'word_length_samples', 'num_samples_before',
       'num_samples_after', 'start_ms', 'end_ms'],
      dtype='object')

In [284]:
munged_excerpt_labels = all_timit[['signal',
                                   'speaker_sex', 'sentence_type',
                                   'sentence_id', 'dialect_region',
                                   'data_split']].loc[good_timit._full_dataset_index.values]
munged_excerpt_labels.reset_index(inplace=True, drop=True)

In [285]:
good_timit.reset_index(inplace=True, drop=True)

In [286]:
new_timit = pd.concat([good_timit, munged_excerpt_labels], axis=1)

In [287]:
new_timit.columns

Index(['start_sample', 'end_sample', 'word', '_full_dataset_index', 'corpus',
       'path', 'source', 'speaker', 'sr', 'signal_length', 'duration_ms',
       'n_channels', 'word_length_samples', 'num_samples_before',
       'num_samples_after', 'start_ms', 'end_ms', 'signal', 'speaker_sex',
       'sentence_type', 'sentence_id', 'dialect_region', 'data_split'],
      dtype='object')

In [288]:
new_timit.head()

,start_sample,end_sample,word,_full_dataset_index,corpus,path,source,speaker,sr,signal_length,...,num_samples_before,num_samples_after,start_ms,end_ms,signal,speaker_sex,sentence_type,sentence_id,dialect_region,data_split
0,12291,15325,money,3,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-si1657,fcjf0,16000,45466,...,12291,30141,0.768188,0.957812,"[[0.00012207403790398877, 9.155552842799158e-0...",f,si,1657,dr1,train
1,11640,23170,programs,15,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdaw0-sx146,fdaw0,16000,42087,...,11640,18917,0.727500,1.448125,"[[0.00027466658528397473, 3.051850947599719e-0...",f,sx,146,dr1,train
2,22327,28680,novel,17,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdaw0-sx326,fdaw0,16000,45978,...,22327,17298,1.395438,1.792500,"[[-6.103701895199438e-05, -0.00015259254737998...",f,sx,326,dr1,train
3,10040,15382,power,18,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdaw0-sx416,fdaw0,16000,41780,...,10040,26398,0.627500,0.961375,"[[-0.00027466658528397473, 0.00015259254737998...",f,sx,416,dr1,train
4,20200,22760,should,28,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fdml0-sx429,fdml0,16000,44237,...,20200,21477,1.262500,1.422500,"[[0.0, 0.0, 0.0, 9.155552842799158e-05, 6.1037...",f,sx,429,dr1,train


In [289]:
## Remove unneeded columns

new_timit.drop(columns=['path', 'corpus', 'start_ms', 'end_ms'], inplace=True)

In [290]:
# rename signal column
new_timit.rename(columns={'signal':'original_signal'}, inplace=True)

In [291]:
new_timit.head()

,start_sample,end_sample,word,_full_dataset_index,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,original_signal,speaker_sex,sentence_type,sentence_id,dialect_region,data_split
0,12291,15325,money,3,train-dr1-fcjf0-si1657,fcjf0,16000,45466,2841.6250,1,3034,12291,30141,"[[0.00012207403790398877, 9.155552842799158e-0...",f,si,1657,dr1,train
1,11640,23170,programs,15,train-dr1-fdaw0-sx146,fdaw0,16000,42087,2630.4375,1,11530,11640,18917,"[[0.00027466658528397473, 3.051850947599719e-0...",f,sx,146,dr1,train
2,22327,28680,novel,17,train-dr1-fdaw0-sx326,fdaw0,16000,45978,2873.6250,1,6353,22327,17298,"[[-6.103701895199438e-05, -0.00015259254737998...",f,sx,326,dr1,train
3,10040,15382,power,18,train-dr1-fdaw0-sx416,fdaw0,16000,41780,2611.2500,1,5342,10040,26398,"[[-0.00027466658528397473, 0.00015259254737998...",f,sx,416,dr1,train
4,20200,22760,should,28,train-dr1-fdml0-sx429,fdml0,16000,44237,2764.8125,1,2560,20200,21477,"[[0.0, 0.0, 0.0, 9.155552842799158e-05, 6.1037...",f,sx,429,dr1,train


In [292]:
new_timit.shape

(1527, 19)

In [297]:
# cut signals so they are all exactly 2 seconds - get endpoints based on word start and end samples 


def cut_signal(row):
    word_dur = row.end_sample - row.start_sample
    remainder = 32000 - word_dur
    edge_shift = remainder // 2
    if remainder%2 == 1: # if odd frames, just shift by one
        start_ix = row.start_sample - (edge_shift + 1) 
    else:
        start_ix = row.start_sample - edge_shift
    end_ix  = row.end_sample + edge_shift
    # need to ix original_signal[0] to get array due to wrapping in timit df's 
    sig = row.original_signal[0][start_ix:end_ix]
    return sig

new_timit['signal'] = new_timit[['start_sample', 'end_sample', 'original_signal']].apply(lambda x: cut_signal(x), axis=1)



In [298]:
new_timit.shape

(1527, 20)

#### Filter excerpts that are cut incorrectly (these have 0 length) - this is a temporary hack

In [299]:
new_timit['signal_len'] = new_timit['signal'].apply(lambda x: len(x))

In [300]:
new_timit = new_timit[new_timit['signal_len'] != 0]

In [333]:
new_timit.reset_index(inplace=True, drop=True)

In [334]:
new_timit.head()

,index,start_sample,end_sample,word,_full_dataset_index,source,speaker,sr,signal_length,duration_ms,...,num_samples_before,num_samples_after,original_signal,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,signal,signal_len
0,1,11640,23170,programs,15,train-dr1-fdaw0-sx146,fdaw0,20000,42087,2630.4375,...,11640,18917,"[[0.00027466658528397473, 3.051850947599719e-0...",f,sx,146,dr1,train,"[2.3069690259136608e-05, 0.0001174022678404248...",32000
1,2,22327,28680,novel,17,train-dr1-fdaw0-sx326,fdaw0,20000,45978,2873.6250,...,22327,17298,"[[-6.103701895199438e-05, -0.00015259254737998...",f,sx,326,dr1,train,"[0.0010651568947912448, 0.007388919911868498, ...",32000
2,4,20200,22760,should,28,train-dr1-fdml0-sx429,fdml0,20000,44237,2764.8125,...,20200,21477,"[[0.0, 0.0, 0.0, 9.155552842799158e-05, 6.1037...",f,sx,429,dr1,train,"[0.00019998189934976397, 0.0006603605168916681...",32000
3,6,16049,23391,social,35,train-dr1-fecd0-sx158,fecd0,20000,41165,2572.8125,...,16049,17774,"[[0.0005188146610919523, 3.051850947599719e-05...",f,sx,158,dr1,train,"[-9.94791987722646e-05, -0.0001146543672654921...",32000
4,7,16041,24520,light,36,train-dr1-fecd0-sx248,fecd0,20000,48845,3052.8125,...,16041,24325,"[[0.0003357036042359691, -3.051850947599719e-0...",f,sx,248,dr1,train,"[0.005607189393168457, 0.003494775608541276, 0...",32000


In [7]:
new_timit.shape

(947, 12)

In [315]:
def resample(x):
    return nnresample.resample(x,5,4) # new sr is 20kHz old is 16kHz, is 5/4 ratio 

new_timit['signal'] = new_timit['signal'].apply(lambda x: resample(x))

In [316]:
new_timit['sr'] = 20000 # update sampling rate column 

In [340]:
new_timit['signal_length'] = new_timit['signal'].apply(lambda x: len(x)) # update signal length column

In [321]:
Audio(new_timit['signal'].iloc[1], rate = 20000)

In [330]:
np.setdiff1d(new_timit.word.unique(), valid_timit_words) # all words contained 

array([], dtype=object)

In [332]:
new_timit.columns

Index(['index', 'start_sample', 'end_sample', 'word', '_full_dataset_index',
       'source', 'speaker', 'sr', 'signal_length', 'duration_ms', 'n_channels',
       'word_length_samples', 'num_samples_before', 'num_samples_after',
       'original_signal', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'signal_len'],
      dtype='object')

In [342]:
# drop unnecessary set of columns 

new_timit.drop(columns=['index', 'start_sample', 'end_sample','duration_ms', 'n_channels',
       'word_length_samples', 'num_samples_before', 'num_samples_after',
       'original_signal', 'signal_len'], inplace=True)

In [343]:
new_timit.head()

,word,_full_dataset_index,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,signal
0,programs,15,train-dr1-fdaw0-sx146,fdaw0,20000,40000,f,sx,146,dr1,train,"[2.3069690259136608e-05, 0.0001174022678404248..."
1,novel,17,train-dr1-fdaw0-sx326,fdaw0,20000,40000,f,sx,326,dr1,train,"[0.0010651568947912448, 0.007388919911868498, ..."
2,should,28,train-dr1-fdml0-sx429,fdml0,20000,40000,f,sx,429,dr1,train,"[0.00019998189934976397, 0.0006603605168916681..."
3,social,35,train-dr1-fecd0-sx158,fecd0,20000,40000,f,sx,158,dr1,train,"[-9.94791987722646e-05, -0.0001146543672654921..."
4,light,36,train-dr1-fecd0-sx248,fecd0,20000,40000,f,sx,248,dr1,train,"[0.005607189393168457, 0.003494775608541276, 0..."


In [346]:
new_timit.shape

(947, 12)

### Add word ix labels from our vocabulary

In [12]:
word_to_ix_map = {word:ix for ix, word in class_map.items()}

In [14]:
new_timit['word_int'] = new_timit['word'].map(word_to_ix_map)

In [18]:
new_timit['word_int'].isnull().values.any() # all words properly mapped if False! 

False

### Save dataframe 


In [2]:
out_path = '/om2/user/imgriff/datasets/timit/timit_wsn_compatible.pdpkl'

In [3]:
from pathlib import Path

In [4]:
path = Path(out_path)

In [351]:
path

PosixPath('/om2/user/imgriff/datasets/timit/timit_wsn_compatible.pdpkl')

In [19]:
# new_timit.to_pickle(path)

In [5]:
new_timit = pd.read_pickle(path)

In [102]:
new_timit.head()

,word,_full_dataset_index,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,signal,word_int
0,programs,15,train-dr1-fdaw0-sx146,fdaw0,20000,40000,f,sx,146,dr1,train,"[2.3069690259136608e-05, 0.0001174022678404248...",552
1,novel,17,train-dr1-fdaw0-sx326,fdaw0,20000,40000,f,sx,326,dr1,train,"[0.0010651568947912448, 0.007388919911868498, ...",461
2,should,28,train-dr1-fdml0-sx429,fdml0,20000,40000,f,sx,429,dr1,train,"[0.00019998189934976397, 0.0006603605168916681...",646
3,social,35,train-dr1-fecd0-sx158,fecd0,20000,40000,f,sx,158,dr1,train,"[-9.94791987722646e-05, -0.0001146543672654921...",659
4,light,36,train-dr1-fecd0-sx248,fecd0,20000,40000,f,sx,248,dr1,train,"[0.005607189393168457, 0.003494775608541276, 0...",393


0.09234841964397407

### Develop `__getitem__` for pytorch dataset class 

In [20]:
index = 10

In [21]:
foreground = new_timit.signal[index]

In [23]:
talker = new_timit.speaker[index]

In [24]:
talker

'fetb0'

In [26]:
cue_ixs = np.where(new_timit.speaker == talker)[0]

In [27]:
cue_ixs

array([ 6,  7,  8,  9, 10])

In [28]:
cue_ixs[cue_ixs != index]

array([6, 7, 8, 9])

In [83]:
cue_ix = np.random.choice(cue_ixs)

In [85]:
fg_cue = new_timit.signal[cue_ix]

(40000,)

In [29]:
bg_ixs = np.where(new_timit.speaker != talker)[0]

#### Test single bg talker selection

In [99]:
talker_ixs = np.random.choice(bg_ixs, size=1, replace=False) # returns array

In [100]:
background_talkers = new_timit.signal[talker_ixs]
background_talkers = np.stack(background_talkers)
print(background_talkers.shape)

(1, 40000)


In [101]:
background_talkers.squeeze().shape

(40000,)

#### Test multi-bg talker selection

In [80]:
talker_ixs = np.random.choice(bg_ixs, size=4, replace=False)


In [81]:
talker_ixs

array([102, 182, 277, 778])

In [82]:
background_talkers = new_timit.signal[talker_ixs]
np.stack(background_talkers).shape

(4, 40000)

#### Most general:  Just use stack and strip singleton dimension in 1 bg talker case